In [1]:
import sys
print(sys.executable)

f:\Study ITU\Research Project\ecpfs\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
from ecpfs import Builder, Index, Metric

print("eCP-FS import successful!")
print("Working directory:", Path.cwd())

eCP-FS import successful!
Working directory: f:\Study ITU\Research Project\ecpfs\mycode


In [3]:
from pathlib import Path
import h5py

# Locate the dataset
data_path = Path.cwd().parent / "data" / "ccnews-nomic-768-normalized.h5"

print("Dataset exists:", data_path.exists())

# Open and inspect the dataset
with h5py.File(data_path, "r") as f:
    print("Train shape:", f["train"].shape)
    print("Test shape:", f["test"].shape)
    print("First 5 values:", f["train"][0, :5])

Dataset exists: True
Train shape: (495328, 768)
Test shape: (1000, 768)
First 5 values: [ 0.00118432  0.04939296 -0.21632339  0.05403711  0.07347868]


In [4]:
import h5py

with h5py.File(data_path, "r") as f:

    print("=== DATASET INFORMATION ===")

    # Display dataset metadata
    for key, value in f.attrs.items():
        print(f"{key}: {value}")

    print("\n=== DATASET NAMES AND STRUCTURE ===")

    # Display every dataset name, shape and data type
    for name in f.keys():
        dataset = f[name]

        print(f"\nName: {name}")
        print(f"Shape: {dataset.shape}")
        print(f"Data type: {dataset.dtype}")

    print("\n=== EXAMPLE VALUES ===")

    print("First train vector:", f["train"][0, :10])
    print("First test vector:", f["test"][0, :10])
    print("First 10 neighbor IDs:", f["neighbors"][0, :10])
    print("First 10 distances:", f["distances"][0, :10])
    print("First average distance:", f["avg_distances"][0])

=== DATASET INFORMATION ===
dimension: 768
distance: normalized
point_type: float

=== DATASET NAMES AND STRUCTURE ===

Name: avg_distances
Shape: (1000,)
Data type: float64

Name: distances
Shape: (1000, 100)
Data type: float64

Name: neighbors
Shape: (1000, 100)
Data type: int64

Name: test
Shape: (1000, 768)
Data type: float32

Name: train
Shape: (495328, 768)
Data type: float32

=== EXAMPLE VALUES ===
First train vector: [ 0.00118432  0.04939296 -0.21632339  0.05403711  0.07347868  0.04265145
  0.0213018   0.05593102  0.03132028 -0.00196485]
First test vector: [-0.01059632  0.00122034 -0.20069289 -0.05815577 -0.01281362 -0.04921297
 -0.04323493 -0.04478858  0.00307377 -0.01775028]
First 10 neighbor IDs: [331646 460655 118234  80550 218017 269547 362345 346445  32329 306328]
First 10 distances: [0.14955997 0.1539191  0.15749824 0.16052967 0.17653155 0.18299526
 0.18959135 0.19473642 0.1959163  0.19612479]
First average distance: 0.4515838623046875


Name	Meaning	Purpose in research
train	495,328 vectors, each with 768 dimensions	Used to build the eCP-FS index
test	1,000 query vectors, each with 768 dimensions	Used to search the index
neighbors	IDs of the 100 true nearest neighbours for each query	Used to calculate recall
distances	Actual distances of those 100 nearest neighbours	Used to evaluate distance results
avg_distances	Average distance from each query to all corpus vectors	Useful for understanding dataset/query characteristics

In [5]:
import numpy as np

with h5py.File(data_path, "r") as f:
    train_vector = f["train"][0]
    test_vector = f["test"][0]

print("Train vector length:", np.linalg.norm(train_vector))
print("Test vector length:", np.linalg.norm(test_vector))

Train vector length: 1.0
Test vector length: 1.0000001


In [6]:
with h5py.File(data_path, "r") as f:
    query = f["test"][0]
    neighbor_id = int(f["neighbors"][0, 0])
    neighbor = f["train"][neighbor_id]
    recorded_distance = f["distances"][0, 0]

# Calculate different distances
euclidean = np.linalg.norm(query - neighbor)
squared_euclidean = euclidean ** 2

cosine_distance = 1 - (
    np.dot(query, neighbor) /
    (np.linalg.norm(query) * np.linalg.norm(neighbor))
)

print("Neighbor ID:", neighbor_id)
print("Recorded distance:", recorded_distance)
print("Euclidean distance:", euclidean)
print("Squared Euclidean:", squared_euclidean)
print("Cosine distance:", cosine_distance)

Neighbor ID: 331646
Recorded distance: 0.14955997467041016
Euclidean distance: 0.54691863
Squared Euclidean: 0.29911998
Cosine distance: 0.14955997


In [7]:
import numpy as np
import h5py

with h5py.File(data_path, "r") as f:
    for name in ["train", "test"]:

        dataset = f[name]
        total_checked = 0
        not_normalized = 0
        max_difference = 0

        # Read 10,000 vectors at a time
        for start in range(0, len(dataset), 10000):
            vectors = dataset[start:start + 10000]

            lengths = np.linalg.norm(vectors, axis=1)

            differences = np.abs(lengths - 1.0)

            not_normalized += np.count_nonzero(
                ~np.isfinite(lengths) | (differences > 0.0001)
            )

            max_difference = max(
                max_difference,
                float(np.max(differences))
            )

            total_checked += len(vectors)

        print(f"\nDataset: {name}")
        print("Total vectors checked:", total_checked)
        print("Vectors not normalized:", not_normalized)
        print("Maximum difference from 1:", max_difference)


Dataset: train
Total vectors checked: 495328
Vectors not normalized: 0
Maximum difference from 1: 1.1920928955078125e-07

Dataset: test
Total vectors checked: 1000
Vectors not normalized: 0
Maximum difference from 1: 1.1920928955078125e-07


In [8]:
print(Builder.__doc__)

Builds a new eCP index in three steps: create the ``Builder``, call
``select_representatives`` or ``select_representatives_custom``, then call
``build``. Every method releases the GIL while it works.

The index is created at ``index_path``. ``levels`` is the number of node
levels below the root, and ``is_normalized`` should be set only if every
embedding is unit-length. ``memory_limit_bytes`` is a target for the build's
memory use rather than a hard cap, and defaults to 80% of system RAM. Leave
``embedding_dtype`` as None to store each file's own dtype; a narrower one
logs a warning. ``rep_chunk_bytes`` is the chunk size for the representative
arrays and ``node_chunk_bytes`` the chunk size for the tree nodes. Measure
zarr read speed at a few chunk sizes on your own data before changing them.


In [9]:
import math
import h5py

# Read dataset information
with h5py.File(data_path, "r") as f:
    N, D = f["train"].shape
    bytes_per_value = f["train"].dtype.itemsize

# Parameters from the paper
V = D * bytes_per_value
C = 128 * 1024
L = 3

# Calculate using the paper's formulas
items_per_cluster = C / V
leaders = (N * V) / C
branching_factor = leaders ** (1 / L)

# Integer target used by the Python implementation
target_cluster_items = round(items_per_cluster)

print("Total vectors (N):", N)
print("Dimensions (D):", D)
print("Vector size (V):", V, "bytes")
print("Desired cluster size (C):", C, "bytes")
print("Index levels (L):", L)

print("\n--- Calculated Parameters ---")
print("Vectors per cluster:", round(items_per_cluster, 2))
print("Estimated cluster leaders:", round(leaders, 2))
print("Estimated branching factor:", round(branching_factor, 2))
print("Target cluster items:", target_cluster_items)

Total vectors (N): 495328
Dimensions (D): 768
Vector size (V): 3072 bytes
Desired cluster size (C): 131072 bytes
Index levels (L): 3

--- Calculated Parameters ---
Vectors per cluster: 42.67
Estimated cluster leaders: 11609.25
Estimated branching factor: 22.64
Target cluster items: 43


In [10]:
import math

# Values from Step 13
N = 495328
target_cluster_items = 43
L = 3

# Calculations used by the implementation
representatives = math.ceil(N / target_cluster_items)
fanout = math.ceil(representatives ** (1 / L))

print("Target items per cluster:", target_cluster_items)
print("Number of representatives:", representatives)
print("Tree levels:", L)
print("Branching factor:", fanout)

Target items per cluster: 43
Number of representatives: 11520
Tree levels: 3
Branching factor: 23


In [11]:
print(Builder.select_representatives.__doc__)

select_representatives(embeddings_file, target_cluster_items, strategy, fallback_batch_rows, grp_name="embeddings")

Picks the representatives the tree is built from, out of the dataset
``grp_name`` in ``embeddings_file`` (``.h5`` or ``.zarr``). ``strategy`` is
"offset" (evenly spaced items) or "random". ``target_cluster_items`` is the
average cluster size to aim for, and ``fallback_batch_rows`` the chunk size
assumed if the file isn't chunked.


In [12]:

import time
import json
from datetime import datetime
from pathlib import Path
from ecpfs import Builder, Metric

# Project paths
project_path = Path.cwd().parent
data_path = project_path / "data" / "ccnews-nomic-768-normalized.h5"
index_path = project_path / "save" / "ccnews_L3_random_43.zarr"

# Prevent accidental overwriting
if index_path.exists():
    raise FileExistsError(f"Index already exists: {index_path}")

index_path.parent.mkdir(parents=True, exist_ok=True)

# Start measuring total build time
start_total = time.perf_counter()

# Step 1: Initialize Builder
start = time.perf_counter()

builder = Builder(
    index_path=str(index_path),
    levels=3,
    metric=Metric.L2,
    is_normalized=True
)

initialization_time = time.perf_counter() - start
print("Builder initialized:", round(initialization_time, 2), "seconds")

# Step 2: Select random representatives
start = time.perf_counter()

builder.select_representatives(
    embeddings_file=str(data_path),
    target_cluster_items=43,
    strategy="random",
    fallback_batch_rows=10000,
    grp_name="train"
)

selection_time = time.perf_counter() - start
print("Representative selection:", round(selection_time, 2), "seconds")

# Step 3: Build the index
start = time.perf_counter()

builder.build(
    embeddings_file=str(data_path),
    fallback_batch_rows=10000,
    grp_name="train"
)

construction_time = time.perf_counter() - start
total_time = time.perf_counter() - start_total

print("\n=== BUILD TIME RESULTS ===")
print("Initialization:", round(initialization_time, 2), "seconds")
print("Selection:", round(selection_time, 2), "seconds")
print("Construction:", round(construction_time, 2), "seconds")
print("Total build time:", round(total_time, 2), "seconds")
print("Total minutes:", round(total_time / 60, 2))

# Save results for research documentation
results = {
    "dataset": data_path.name,
    "git_commit": "7a0b7d7",
    "levels": 3,
    "metric": "L2",
    "is_normalized": True,
    "strategy": "random",
    "target_cluster_items": 43,
    "fallback_batch_rows": 10000,
    "initialization_seconds": initialization_time,
    "selection_seconds": selection_time,
    "construction_seconds": construction_time,
    "total_seconds": total_time
}

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = Path.cwd() / f"build_timing_{timestamp}.json"

with open(log_path, "w") as f:
    json.dump(results, f, indent=4)

print("\nTiming results saved to:", log_path)


Builder initialized: 0.12 seconds
Representative selection: 89.84 seconds

=== BUILD TIME RESULTS ===
Initialization: 0.12 seconds
Selection: 89.84 seconds
Construction: 281.24 seconds
Total build time: 371.2 seconds
Total minutes: 6.19

Timing results saved to: f:\Study ITU\Research Project\ecpfs\mycode\build_timing_20261025_223334.json


Builder initialization (0.12 seconds): The Builder creates the initial index structure and stores the index configuration.

Representative selection (89.84 seconds): The implementation randomly selects representatives from the training dataset. These representatives are used to construct the hierarchical index.

Index construction (281.24 seconds): The implementation organizes the dataset into the hierarchical structure and stores the index using Zarr.

Total time (371.20 seconds): The complete measured index-building procedure took approximately 6 minutes and 11 seconds.

In [13]:
from pathlib import Path
from ecpfs import Index
import time

# Locate the previously built index
index_path = Path.cwd().parent / "save" / "ccnews_L3_random_43.zarr"

print("Index exists:", index_path.exists())
print("Index location:", index_path)

# Measure index loading time
start = time.perf_counter()

with Index(str(index_path)) as index:
    load_time = time.perf_counter() - start
    print("\nIndex loaded successfully!")
    print("Loading time:", round(load_time, 3), "seconds")

print("Index closed successfully!")

Index exists: True
Index location: f:\Study ITU\Research Project\ecpfs\save\ccnews_L3_random_43.zarr

Index loaded successfully!
Loading time: 0.106 seconds
Index closed successfully!


In [14]:
from pathlib import Path

index_path = Path.cwd().parent / "save" / "ccnews_L3_random_43.zarr"

print("=== eCP-FS Index Structure ===")

for item in sorted(index_path.iterdir()):
    item_type = "Folder" if item.is_dir() else "File"
    print(f"{item_type}: {item.name}")

=== eCP-FS Index Structure ===
Folder: index_root
Folder: info
Folder: lvl_1
Folder: lvl_2
Folder: lvl_3
Folder: rep_embeddings
Folder: rep_item_ids
File: zarr.json


In [15]:
from pathlib import Path

info_path = index_path / "info"

print("=== INDEX INFO STRUCTURE ===")

for item in sorted(info_path.iterdir()):
    print(
        "Folder:" if item.is_dir() else "File:",
        item.name
    )

=== INDEX INFO STRUCTURE ===
Folder: format_version
Folder: is_normalized
Folder: levels
Folder: metric
Folder: next_item_id
Folder: total_items
File: zarr.json


In [16]:
import importlib.util

# Check whether Zarr is available
if importlib.util.find_spec("zarr") is None:
    print("Zarr Python package is not installed.")
else:
    import zarr

    # Open the existing index without modifying it
    root = zarr.open_group(str(index_path), mode="r")

    print("=== ACTUAL INDEX METADATA ===")

    for name in [
        "format_version",
        "is_normalized",
        "levels",
        "metric",
        "next_item_id",
        "total_items"
    ]:
        value = root[f"info/{name}"][()]
        print(f"{name}: {value}")

=== ACTUAL INDEX METADATA ===
format_version: 1
is_normalized: True
levels: 3
metric: L2
next_item_id: 495328
total_items: 495328


In [17]:
import zarr
import numpy as np

root = zarr.open_group(str(index_path), mode="r")

# Read stored representative IDs and embedding information
rep_ids = root["rep_item_ids"][:]
rep_embeddings = root["rep_embeddings"]

print("=== REPRESENTATIVE VERIFICATION ===")
print("Number of representatives:", len(rep_ids))
print("Representative embedding shape:", rep_embeddings.shape)
print("Number of unique IDs:", len(np.unique(rep_ids)))
print("First 10 representative IDs:", rep_ids[:10])
print("Minimum ID:", rep_ids.min())
print("Maximum ID:", rep_ids.max())

=== REPRESENTATIVE VERIFICATION ===
Number of representatives: 11520
Representative embedding shape: (11520, 768)
Number of unique IDs: 11520
First 10 representative IDs: [134 145 159 162 183 224 261 313 339 524]
Minimum ID: 134
Maximum ID: 495284


In [18]:
import zarr

root = zarr.open_group(str(index_path), mode="r")

# Inspect root representatives
root_embeddings = root["index_root/embeddings"]

print("=== ROOT NODE INFORMATION ===")
print("Root embeddings shape:", root_embeddings.shape)
print("Number of root representatives:", root_embeddings.shape[0])
print("Dimensions:", root_embeddings.shape[1])

# Inspect first-level node folders
level1_path = index_path / "lvl_1"

nodes = [p.name for p in level1_path.iterdir() if p.is_dir()]

print("\n=== LEVEL 1 INFORMATION ===")
print("Number of entries:", len(nodes))
print("First 10 entries:", sorted(nodes)[:10])

=== ROOT NODE INFORMATION ===
Root embeddings shape: (23, 768)
Number of root representatives: 23
Dimensions: 768

=== LEVEL 1 INFORMATION ===
Number of entries: 23
First 10 entries: ['node_0', 'node_1', 'node_10', 'node_11', 'node_12', 'node_13', 'node_14', 'node_15', 'node_16', 'node_17']


In [19]:
# Access the first node at level 1
node = root["lvl_1/node_0"]

print("=== LEVEL 1: NODE 0 ===")

for name in node.keys():
    item = node[name]

    print("Name:", name)
    print("Shape:", item.shape)
    print("Data type:", item.dtype)
    print()

=== LEVEL 1: NODE 0 ===
Name: node_ids
Shape: (39,)
Data type: uint32

Name: embeddings
Shape: (39, 768)
Data type: float32

Name: border
Shape: (2,)
Data type: float32



In [20]:
# Read the child IDs of Level 1, Node 0
child_ids = root["lvl_1/node_0/node_ids"][:]

print("=== LEVEL 1 TO LEVEL 2 CONNECTION ===")

print("Total child IDs:", len(child_ids))
print("First 10 child IDs:", child_ids[:10])

print("\nChecking first 5 child nodes:")

for child_id in child_ids[:5]:
    child_path = index_path / "lvl_2" / f"node_{int(child_id)}"

    print(
        f"node_{child_id}",
        "exists:", child_path.is_dir()
    )

print("\nBorder values:", root["lvl_1/node_0/border"][:])

=== LEVEL 1 TO LEVEL 2 CONNECTION ===
Total child IDs: 39
First 10 child IDs: [  0  24  46  57  64  90  96  99 105 127]

Checking first 5 child nodes:
node_0 exists: True
node_24 exists: True
node_46 exists: True
node_57 exists: True
node_64 exists: True

Border values: [0. 0.]


In [21]:
# Open Level 2, Node 0
node = root["lvl_2/node_0"]

print("=== LEVEL 2: NODE 0 ===")

# Examine stored arrays
for name in node.keys():
    item = node[name]
    print(name, "Shape:", item.shape)

# Read references to Level 3
child_ids = node["node_ids"][:]

print("\nTotal Level 3 child references:", len(child_ids))
print("First 10 child IDs:", child_ids[:10])

# Verify first five referenced nodes
print("\n=== CHECK LEVEL 3 CONNECTIONS ===")

for child_id in child_ids[:5]:
    child_path = index_path / "lvl_3" / f"node_{int(child_id)}"
    print(f"node_{child_id} exists:", child_path.is_dir())

=== LEVEL 2: NODE 0 ===
embeddings Shape: (35, 768)
border Shape: (2,)
node_ids Shape: (35,)

Total Level 3 child references: 35
First 10 child IDs: [   0 1158 1642 1688 1867 2213 2233 2287 2405 3878]

=== CHECK LEVEL 3 CONNECTIONS ===
node_0 exists: True
node_1158 exists: True
node_1642 exists: True
node_1688 exists: True
node_1867 exists: True


In [22]:
# Open the first leaf cluster
leaf = root["lvl_3/node_0"]

print("=== LEVEL 3: LEAF NODE 0 ===")

# Inspect stored arrays
for name in leaf.keys():
    item = leaf[name]
    print(name, "Shape:", item.shape, "Dtype:", item.dtype)

# Examine the actual item IDs
item_ids = leaf["item_ids"][:]

print("\nNumber of vectors:", len(item_ids))
print("First 10 item IDs:", item_ids[:10])

# Examine the first stored vector
print("\nFirst vector (5 dimensions):")
print(leaf["embeddings"][0, :5])

=== LEVEL 3: LEAF NODE 0 ===
border Shape: (2,) Dtype: float32
embeddings Shape: (186, 768) Dtype: float32
item_ids Shape: (186,) Dtype: uint32

Number of vectors: 186
First 10 item IDs: [  134   188  1146  2879  4269  6583  8429 11814 13891 15103]

First vector (5 dimensions):
[ 0.04047636  0.0185555  -0.20522392  0.03536798  0.06950272]


In [24]:

import time
import h5py
from ecpfs import Index

# Experiment settings
query_number = 0
k = 10
search_exp = 4

# Load query and ground truth
with h5py.File(data_path, "r") as f:
    query = f["test"][query_number]
    ground_truth = f["neighbors"][query_number, :k]

# Load index and measure ONLY search time
with Index(str(index_path)) as index:

    start = time.perf_counter()

    results, query_id = index.new_search(
        query=query,
        k=k,
        search_exp=search_exp,
        max_increments=0,
        exclude_vec=[]
    )

    search_time = time.perf_counter() - start

# Calculate Recall@10
returned_ids = [int(item_id) for distance, item_id in results]

matches = set(returned_ids) & set(ground_truth.tolist())
recall = len(matches) / k

# Display experiment results
print("=== eCP-FS SINGLE QUERY EXPERIMENT ===")
print("Query number:", query_number)
print("k:", k)
print("Search expansion:", search_exp)

print("\n--- Search Results ---")
for rank, (distance, item_id) in enumerate(results, 1):
    print(f"{rank}. ID: {item_id}, L2 Distance: {distance:.6f}")

print("\n--- Performance ---")
print(f"Search time: {search_time:.6f} seconds")
print(f"Search latency: {search_time * 1000:.3f} ms")
print(f"Correct neighbours: {len(matches)} / {k}")
print(f"Recall@{k}: {recall:.4f}")
print(f"Recall percentage: {recall * 100:.2f}%")


=== eCP-FS SINGLE QUERY EXPERIMENT ===
Query number: 0
k: 10
Search expansion: 4

--- Search Results ---
1. ID: 460655, L2 Distance: 0.554832
2. ID: 80550, L2 Distance: 0.566621
3. ID: 218017, L2 Distance: 0.594191
4. ID: 155727, L2 Distance: 0.654863
5. ID: 211936, L2 Distance: 0.664938
6. ID: 390292, L2 Distance: 0.666323
7. ID: 300429, L2 Distance: 0.667339
8. ID: 203159, L2 Distance: 0.671886
9. ID: 60611, L2 Distance: 0.672206
10. ID: 321847, L2 Distance: 0.672371

--- Performance ---
Search time: 0.036060 seconds
Search latency: 36.060 ms
Correct neighbours: 3 / 10
Recall@10: 0.3000
Recall percentage: 30.00%


In [25]:

import time
import csv
import numpy as np
import h5py
from datetime import datetime
from pathlib import Path
from ecpfs import Index

# ==================================
# 1. EXPERIMENT CONFIGURATION
# ==================================

k = 10
search_exp = 2
max_increments = 0

# Load all test queries and ground truth
with h5py.File(data_path, "r") as f:
    queries = f["test"][:]
    ground_truth = f["neighbors"][:, :k]

total_queries = len(queries)
experiment_results = []

print("Starting eCP-FS benchmark...")
print("Total queries:", total_queries)
print("Search expansion:", search_exp)
print("k:", k)

# ==================================
# 2. RUN ALL QUERIES
# ==================================

with Index(str(index_path)) as index:

    for query_number in range(total_queries):

        query = queries[query_number]

        # Measure search time only
        start = time.perf_counter()

        results, query_id = index.new_search(
            query=query,
            k=k,
            search_exp=search_exp,
            max_increments=max_increments,
            exclude_vec=[]
        )

        search_time = time.perf_counter() - start

        # Extract returned neighbour IDs
        returned_ids = {
            int(item_id)
            for distance, item_id in results
        }

        # Calculate Recall@10
        exact_ids = set(ground_truth[query_number].tolist())
        correct = len(returned_ids & exact_ids)
        recall = correct / k

        # Store individual query results
        experiment_results.append({
            "query": query_number,
            "search_time_ms": search_time * 1000,
            "correct_neighbors": correct,
            "returned_neighbors": len(results),
            "recall": recall
        })

# ==================================
# 3. CALCULATE OVERALL PERFORMANCE
# ==================================

latencies = np.array([
    r["search_time_ms"]
    for r in experiment_results
])

recalls = np.array([
    r["recall"]
    for r in experiment_results
])

total_search_seconds = np.sum(latencies) / 1000
qps = total_queries / total_search_seconds

# ==================================
# 4. DISPLAY EACH QUERY RESULT
# ==================================

print("\n=== INDIVIDUAL QUERY RESULTS ===")
print(f"{'Query':<10} {'Latency (ms)':<18} {'Recall@10':<12}")

for r in experiment_results:
    print(
        f"{r['query']:<10} "
        f"{r['search_time_ms']:<18.3f} "
        f"{r['recall']:<12.2f}"
    )

# ==================================
# 5. DISPLAY OVERALL SUMMARY
# ==================================

print("\n=== BENCHMARK SUMMARY ===")
print("Total queries:", total_queries)
print("Search expansion:", search_exp)
print("k:", k)
print(f"Average Recall@10: {np.mean(recalls):.4f}")
print(f"Average Recall (%): {np.mean(recalls)*100:.2f}%")
print(f"Average latency: {np.mean(latencies):.3f} ms")
print(f"Median latency: {np.median(latencies):.3f} ms")
print(f"P95 latency: {np.percentile(latencies,95):.3f} ms")
print(f"QPS: {qps:.2f}")
print(f"Total search time: {total_search_seconds:.3f} seconds")

# ==================================
# 6. SAVE ALL RESULTS TO CSV
# ==================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_path = Path.cwd() / (
    f"ecpfs_exp{search_exp}_k{k}_{timestamp}.csv"
)

with open(csv_path, "w", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=experiment_results[0].keys()
    )
    writer.writeheader()
    writer.writerows(experiment_results)

print("\nResults saved to:", csv_path)


Starting eCP-FS benchmark...
Total queries: 1000
Search expansion: 2
k: 10

=== INDIVIDUAL QUERY RESULTS ===
Query      Latency (ms)       Recall@10   
0          17.687             0.10        
1          99.117             0.20        
2          122.140            0.00        
3          140.601            0.00        
4          108.942            0.30        
5          82.509             0.00        
6          88.028             0.00        
7          102.141            0.30        
8          70.079             1.00        
9          177.125            0.70        
10         171.516            0.00        
11         112.263            0.70        
12         133.931            0.60        
13         117.200            0.40        
14         154.519            0.20        
15         160.424            0.10        
16         101.365            0.50        
17         181.039            0.00        
18         112.586            0.20        
19         109.804            0

In [26]:

import time
import csv
import numpy as np
import h5py
from datetime import datetime
from pathlib import Path
from ecpfs import Index

# ==================================
# 1. EXPERIMENT CONFIGURATION
#increased expansion to 5 for better recall
# ==================================

k = 10
search_exp = 5
max_increments = 0

# Load all test queries and ground truth
with h5py.File(data_path, "r") as f:
    queries = f["test"][:]
    ground_truth = f["neighbors"][:, :k]

total_queries = len(queries)
experiment_results = []

print("Starting eCP-FS benchmark...")
print("Total queries:", total_queries)
print("Search expansion:", search_exp)
print("k:", k)

# ==================================
# 2. RUN ALL QUERIES
# ==================================

with Index(str(index_path)) as index:

    for query_number in range(total_queries):

        query = queries[query_number]

        # Measure search time only
        start = time.perf_counter()

        results, query_id = index.new_search(
            query=query,
            k=k,
            search_exp=search_exp,
            max_increments=max_increments,
            exclude_vec=[]
        )

        search_time = time.perf_counter() - start

        # Extract returned neighbour IDs
        returned_ids = {
            int(item_id)
            for distance, item_id in results
        }

        # Calculate Recall@10
        exact_ids = set(ground_truth[query_number].tolist())
        correct = len(returned_ids & exact_ids)
        recall = correct / k

        # Store individual query results
        experiment_results.append({
            "query": query_number,
            "search_time_ms": search_time * 1000,
            "correct_neighbors": correct,
            "returned_neighbors": len(results),
            "recall": recall
        })

# ==================================
# 3. CALCULATE OVERALL PERFORMANCE
# ==================================

latencies = np.array([
    r["search_time_ms"]
    for r in experiment_results
])

recalls = np.array([
    r["recall"]
    for r in experiment_results
])

total_search_seconds = np.sum(latencies) / 1000
qps = total_queries / total_search_seconds

# ==================================
# 4. DISPLAY EACH QUERY RESULT
# ==================================

print("\n=== INDIVIDUAL QUERY RESULTS ===")
print(f"{'Query':<10} {'Latency (ms)':<18} {'Recall@10':<12}")

for r in experiment_results:
    print(
        f"{r['query']:<10} "
        f"{r['search_time_ms']:<18.3f} "
        f"{r['recall']:<12.2f}"
    )

# ==================================
# 5. DISPLAY OVERALL SUMMARY
# ==================================

print("\n=== BENCHMARK SUMMARY ===")
print("Total queries:", total_queries)
print("Search expansion:", search_exp)
print("k:", k)
print(f"Average Recall@10: {np.mean(recalls):.4f}")
print(f"Average Recall (%): {np.mean(recalls)*100:.2f}%")
print(f"Average latency: {np.mean(latencies):.3f} ms")
print(f"Median latency: {np.median(latencies):.3f} ms")
print(f"P95 latency: {np.percentile(latencies,95):.3f} ms")
print(f"QPS: {qps:.2f}")
print(f"Total search time: {total_search_seconds:.3f} seconds")

# ==================================
# 6. SAVE ALL RESULTS TO CSV
# ==================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_path = Path.cwd() / (
    f"ecpfs_exp{search_exp}_k{k}_{timestamp}.csv"
)

with open(csv_path, "w", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=experiment_results[0].keys()
    )
    writer.writeheader()
    writer.writerows(experiment_results)

print("\nResults saved to:", csv_path)


Starting eCP-FS benchmark...
Total queries: 1000
Search expansion: 5
k: 10

=== INDIVIDUAL QUERY RESULTS ===
Query      Latency (ms)       Recall@10   
0          55.946             0.30        
1          155.918            0.20        
2          65.098             0.00        
3          116.587            0.00        
4          67.421             0.40        
5          141.769            0.00        
6          145.882            0.00        
7          144.337            0.30        
8          158.039            1.00        
9          120.730            0.80        
10         101.840            0.10        
11         129.617            0.70        
12         81.021             0.70        
13         66.910             0.60        
14         114.371            0.30        
15         127.216            0.10        
16         170.311            0.50        
17         251.944            0.00        
18         145.473            0.20        
19         163.921            0

In [27]:
import csv
from pathlib import Path

# Compare previous benchmark experiments
for expansion in [2, 5]:

    # Find the saved CSV file
    files = sorted(
        Path.cwd().glob(f"ecpfs_exp{expansion}_k10_*.csv")
    )

    # Read the most recent file
    with open(files[-1], "r") as f:
        reader = csv.DictReader(f)
        recalls = [float(row["recall"]) for row in reader]

    # Calculate average recall
    average_recall = sum(recalls) / len(recalls)

    print(f"Search expansion: {expansion}")
    print(f"Total queries: {len(recalls)}")
    print(f"Average Recall@10: {average_recall:.4f}")
    print(f"Average Recall (%): {average_recall * 100:.2f}%")
    print("-" * 35)

Search expansion: 2
Total queries: 1000
Average Recall@10: 0.2451
Average Recall (%): 24.51%
-----------------------------------
Search expansion: 5
Total queries: 1000
Average Recall@10: 0.3134
Average Recall (%): 31.34%
-----------------------------------


In [29]:
from ecpfs import Index

with Index(str(index_path)) as index:
    print("Index opened successfully")

print("Index closed successfully")

Index opened successfully
Index closed successfully


In [30]:

import time
import csv
import h5py
from pathlib import Path
from ecpfs import Index

# ==================================
# 1. EXPERIMENT CONFIGURATION
# ==================================

k = 10
search_exp = 5
max_increments = 0

# Load queries and ground truth before timing
with h5py.File(data_path, "r") as f:
    queries = {
        0: f["test"][0],
        1: f["test"][1]
    }

    ground_truth = {
        0: set(f["neighbors"][0, :k].tolist()),
        1: set(f["neighbors"][1, :k].tolist())
    }

experiment_results = []

# ==================================
# 2. RUN SEARCHES WITH SAME INDEX
# ==================================

search_sequence = [
    ("Query 0 - First Search", 0),
    ("Query 1 - Second Search", 1),
    ("Query 0 - Repeated Search", 0)
]

with Index(str(index_path)) as index:

    for stage, query_number in search_sequence:

        # Measure only search execution
        start = time.perf_counter()

        results, query_id = index.new_search(
            query=queries[query_number],
            k=k,
            search_exp=search_exp,
            max_increments=max_increments,
            exclude_vec=[]
        )

        latency_ms = (time.perf_counter() - start) * 1000

        # Calculate Recall@10
        returned_ids = {int(item_id) for distance, item_id in results}
        correct = len(returned_ids & ground_truth[query_number])
        recall = correct / k

        experiment_results.append({
            "stage": stage,
            "query": query_number,
            "latency_ms": latency_ms,
            "correct_neighbors": correct,
            "recall": recall
        })

# ==================================
# 3. DISPLAY RESULTS
# ==================================

print("=== COLD VS WARM CACHE EXPERIMENT ===")
print("Search expansion:", search_exp)
print("k:", k)

for r in experiment_results:
    print(f"\n{r['stage']}")
    print(f"Search latency: {r['latency_ms']:.3f} ms")
    print(f"Correct neighbours: {r['correct_neighbors']}/{k}")
    print(f"Recall@10: {r['recall']:.2f} ({r['recall']*100:.1f}%)")

# Compare the two executions of Query 0
first_time = experiment_results[0]["latency_ms"]
repeat_time = experiment_results[2]["latency_ms"]

print("\n=== QUERY 0 COMPARISON ===")
print(f"First search: {first_time:.3f} ms")
print(f"Repeated search: {repeat_time:.3f} ms")
print(f"Speedup factor: {first_time / repeat_time:.2f}x")
print(f"Time saved: {first_time - repeat_time:.3f} ms")

# ==================================
# 4. SAVE RESULTS TO CSV
# ==================================

csv_path = Path.cwd() / "ecpfs_cold_warm_experiment.csv"

with open(csv_path, "w", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=experiment_results[0].keys()
    )
    writer.writeheader()
    writer.writerows(experiment_results)

print("\nResults saved to:", csv_path)
print("Index closed successfully.")


=== COLD VS WARM CACHE EXPERIMENT ===
Search expansion: 5
k: 10

Query 0 - First Search
Search latency: 350.120 ms
Correct neighbours: 3/10
Recall@10: 0.30 (30.0%)

Query 1 - Second Search
Search latency: 258.467 ms
Correct neighbours: 2/10
Recall@10: 0.20 (20.0%)

Query 0 - Repeated Search
Search latency: 16.445 ms
Correct neighbours: 3/10
Recall@10: 0.30 (30.0%)

=== QUERY 0 COMPARISON ===
First search: 350.120 ms
Repeated search: 16.445 ms
Speedup factor: 21.29x
Time saved: 333.675 ms

Results saved to: f:\Study ITU\Research Project\ecpfs\mycode\ecpfs_cold_warm_experiment.csv
Index closed successfully.


In [31]:

import time
import csv
import h5py
from datetime import datetime
from pathlib import Path
from ecpfs import Index

# =====================================
# 1. EXPERIMENT CONFIGURATION
# =====================================

query_number = 0
k = 10
search_exp = 5
max_increments = 0
pages = 3

# Load query and ground truth
with h5py.File(data_path, "r") as f:
    query = f["test"][query_number]
    ground_truth = f["neighbors"][query_number, :30]

all_returned_ids = []
experiment_results = []
total_search_ms = 0

# =====================================
# 2. INCREMENTAL SEARCH
# =====================================

with Index(str(index_path)) as index:

    for page in range(1, pages + 1):

        start = time.perf_counter()

        if page == 1:
            # Start a new query
            results, query_id = index.new_search(
                query=query,
                k=k,
                search_exp=search_exp,
                max_increments=max_increments,
                exclude_vec=[]
            )
        else:
            # Continue the SAME query
            results = index.get_next_k_items(
                query_id=query_id,
                k=k,
                search_exp=search_exp,
                max_increments=max_increments,
                exclude_vec=[]
            )

        latency_ms = (time.perf_counter() - start) * 1000
        total_search_ms += latency_ms

        # Collect newly returned IDs
        page_ids = [int(item_id) for _, item_id in results]
        all_returned_ids.extend(page_ids)

        # Calculate cumulative Recall@K
        cumulative_k = page * k
        exact_ids = set(ground_truth[:cumulative_k].tolist())

        correct = len(set(all_returned_ids) & exact_ids)
        recall = correct / cumulative_k

        # Store measurements
        experiment_results.append({
            "stage": page,
            "query": query_number,
            "requested_total_k": cumulative_k,
            "returned_this_stage": len(results),
            "stage_latency_ms": latency_ms,
            "cumulative_latency_ms": total_search_ms,
            "correct_neighbors": correct,
            "cumulative_recall": recall,
            "returned_ids": ";".join(map(str, page_ids))
        })

        print(f"\n=== STAGE {page} ===")
        print("New IDs:", page_ids)
        print(f"Stage latency: {latency_ms:.3f} ms")
        print(f"Cumulative latency: {total_search_ms:.3f} ms")
        print(f"Correct neighbours: {correct}/{cumulative_k}")
        print(f"Recall@{cumulative_k}: {recall * 100:.2f}%")

# =====================================
# 3. FINAL SUMMARY
# =====================================

print("\n=== INCREMENTAL SEARCH SUMMARY ===")

for r in experiment_results:
    print(
        f"Stage {r['stage']} | "
        f"Time: {r['stage_latency_ms']:.3f} ms | "
        f"Recall@{r['requested_total_k']}: "
        f"{r['cumulative_recall'] * 100:.2f}%"
    )

print(f"\nTotal search time: {total_search_ms:.3f} ms")
print("Total unique neighbours:", len(set(all_returned_ids)))

# =====================================
# 4. SAVE RESULTS
# =====================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_path = Path.cwd() / (
    f"ecpfs_incremental_query{query_number}_{timestamp}.csv"
)

with open(csv_path, "w", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=experiment_results[0].keys()
    )
    writer.writeheader()
    writer.writerows(experiment_results)

print("\nResults saved to:", csv_path)
print("Index closed successfully.")



=== STAGE 1 ===
New IDs: [460655, 80550, 218017, 155727, 211936, 390292, 300429, 203159, 60611, 321847]
Stage latency: 39.001 ms
Cumulative latency: 39.001 ms
Correct neighbours: 3/10
Recall@10: 30.00%

=== STAGE 2 ===
New IDs: [333404, 347460, 380357, 297948, 199277, 24141, 31726, 213458, 424806, 253234]
Stage latency: 1.104 ms
Cumulative latency: 40.106 ms
Correct neighbours: 3/20
Recall@20: 15.00%

=== STAGE 3 ===
New IDs: [178948, 413850, 59796, 218183, 301501, 278125, 404107, 53367, 472304, 222742]
Stage latency: 0.043 ms
Cumulative latency: 40.149 ms
Correct neighbours: 4/30
Recall@30: 13.33%

=== INCREMENTAL SEARCH SUMMARY ===
Stage 1 | Time: 39.001 ms | Recall@10: 30.00%
Stage 2 | Time: 1.104 ms | Recall@20: 15.00%
Stage 3 | Time: 0.043 ms | Recall@30: 13.33%

Total search time: 40.149 ms
Total unique neighbours: 30

Results saved to: f:\Study ITU\Research Project\ecpfs\mycode\ecpfs_incremental_query0_20261025_233959.csv
Index closed successfully.


In [32]:

import time
import csv
import h5py
from pathlib import Path
from datetime import datetime
from ecpfs import Index

# =====================================
# 1. EXPERIMENT CONFIGURATION
# =====================================

query_number = 0
k = 10
total_k = 30
search_exp = 5
max_increments = 0

with h5py.File(data_path, "r") as f:
    query = f["test"][query_number]
    ground_truth = f["neighbors"][query_number, :30]

records = []

# =====================================
# 2. OPEN THE SAME INDEX
# =====================================

with Index(str(index_path)) as index:

    # Warm up both methods before timing
    index.new_search(
        query=query, k=30, search_exp=search_exp,
        max_increments=0, exclude_vec=[]
    )

    _, warm_id = index.new_search(
        query=query, k=10, search_exp=search_exp,
        max_increments=0, exclude_vec=[]
    )

    for _ in range(2):
        index.get_next_k_items(
            query_id=warm_id, k=10,
            search_exp=search_exp,
            max_increments=0, exclude_vec=[]
        )

    # =================================
    # METHOD A: DIRECT SEARCH
    # =================================

    start = time.perf_counter()

    direct_results, direct_query_id = index.new_search(
        query=query,
        k=30,
        search_exp=search_exp,
        max_increments=max_increments,
        exclude_vec=[]
    )

    direct_time = (time.perf_counter() - start) * 1000

    direct_ids = {int(item_id) for _, item_id in direct_results}

    direct_correct = len(
        direct_ids & set(ground_truth.tolist())
    )

    direct_recall = direct_correct / 30

    records.append({
        "method": "Direct",
        "stage": 1,
        "total_k": 30,
        "stage_time_ms": direct_time,
        "cumulative_time_ms": direct_time,
        "recall": direct_recall
    })

    # =================================
    # METHOD B: INCREMENTAL SEARCH
    # =================================

    incremental_ids = []
    incremental_total_time = 0
    incremental_results = []

    for stage in range(1, 4):

        start = time.perf_counter()

        if stage == 1:
            results, inc_query_id = index.new_search(
                query=query, k=10,
                search_exp=search_exp,
                max_increments=max_increments,
                exclude_vec=[]
            )
        else:
            results = index.get_next_k_items(
                query_id=inc_query_id,
                k=10,
                search_exp=search_exp,
                max_increments=max_increments,
                exclude_vec=[]
            )

        stage_time = (time.perf_counter() - start) * 1000

        incremental_total_time += stage_time

        incremental_ids.extend(
            int(item_id) for _, item_id in results
        )

        current_k = stage * 10
        correct = len(
            set(incremental_ids) &
            set(ground_truth[:current_k].tolist())
        )

        recall = correct / current_k

        incremental_results.append({
            "stage": stage,
            "time": stage_time,
            "cumulative_time": incremental_total_time,
            "recall": recall
        })

        records.append({
            "method": "Incremental",
            "stage": stage,
            "total_k": current_k,
            "stage_time_ms": stage_time,
            "cumulative_time_ms": incremental_total_time,
            "recall": recall
        })

# =====================================
# 3. DISPLAY RESULTS
# =====================================

print("=== DIRECT SEARCH (30 NN) ===")
print(f"Neighbours returned: {len(direct_ids)}")
print(f"Search time: {direct_time:.3f} ms")
print(f"Recall@30: {direct_recall * 100:.2f}%")

print("\n=== INCREMENTAL SEARCH (10+10+10) ===")

for r in incremental_results:
    print(
        f"Stage {r['stage']} | "
        f"Time: {r['time']:.3f} ms | "
        f"Recall@{r['stage']*10}: {r['recall']*100:.2f}%"
    )

incremental_correct = len(
    set(incremental_ids) &
    set(ground_truth.tolist())
)
incremental_recall = incremental_correct / 30

print("\n=== FINAL COMPARISON ===")

print(f"{'Metric':<25} {'Direct':<15} {'Incremental':<15}")
print("-" * 55)

print(
    f"{'Total latency (ms)':<25} "
    f"{direct_time:<15.3f} "
    f"{incremental_total_time:<15.3f}"
)

print(
    f"{'Recall@30 (%)':<25} "
    f"{direct_recall*100:<15.2f} "
    f"{incremental_recall*100:<15.2f}"
)

print(
    f"{'Unique neighbours':<25} "
    f"{len(direct_ids):<15} "
    f"{len(set(incremental_ids)):<15}"
)

# =====================================
# 4. SAVE RESULTS
# =====================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

csv_path = Path.cwd() / (
    f"ecpfs_direct_vs_incremental_{timestamp}.csv"
)

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=records[0].keys())
    writer.writeheader()
    writer.writerows(records)

print("\nResults saved to:", csv_path)
print("Index closed successfully.")


=== DIRECT SEARCH (30 NN) ===
Neighbours returned: 30
Search time: 8.002 ms
Recall@30: 13.33%

=== INCREMENTAL SEARCH (10+10+10) ===
Stage 1 | Time: 9.825 ms | Recall@10: 30.00%
Stage 2 | Time: 0.061 ms | Recall@20: 15.00%
Stage 3 | Time: 0.049 ms | Recall@30: 13.33%

=== FINAL COMPARISON ===
Metric                    Direct          Incremental    
-------------------------------------------------------
Total latency (ms)        8.002           9.935          
Recall@30 (%)             13.33           13.33          
Unique neighbours         30              30             

Results saved to: f:\Study ITU\Research Project\ecpfs\mycode\ecpfs_direct_vs_incremental_20261025_234447.csv
Index closed successfully.
